In [1]:
import pandas as pd
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
import os
from scipy.signal import argrelextrema

In [2]:
airgo_research_dir = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/AirGo/airgo_research/'
airgo_features_dir = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/AirGo/airgo_features/'

In [11]:

files = os.listdir(airgo_research_dir)
# read in airgo:
file = files[50]
airgo_data = pd.read_csv(os.path.join(airgo_research_dir, file))
airgo_data['DateTime'] = pd.to_datetime(airgo_data['DateTime'], infer_datetime_format=1)
airgo_data.set_index('DateTime', inplace=True)
# airgo_data = airgo_data.iloc[:100000,:]
print('tmpsetting)')

tmpsetting)


In [12]:
airgo_data.head()

,accX,accY,accZ,Band,CRCStatus
DateTime,,,,,
2019-04-23 16:57:58.010,-5.13,6.68,-7.40,2341,NaN
2019-04-23 16:57:58.110,-6.32,3.96,-5.59,2294,NaN
2019-04-23 16:57:58.210,-3.76,3.85,-7.62,2263,NaN
2019-04-23 16:57:58.310,-4.47,2.54,-7.93,2263,NaN
2019-04-23 16:57:58.410,-3.97,2.05,-8.90,2257,NaN


In [13]:
airgo_data['deriv1'] = np.concatenate([[0],np.diff(airgo_data['Band'])])
airgo_data['deriv2'] = np.concatenate([[0],np.diff(airgo_data['deriv1'])])

In [14]:
airgo_data.iloc[5:15]

,accX,accY,accZ,Band,CRCStatus,deriv1,deriv2
DateTime,,,,,,,
2019-04-23 16:57:58.510,-3.49,2.49,-8.13,2255,NaN,-2,4
2019-04-23 16:57:58.610,-3.00,1.90,-9.16,2265,NaN,10,12
2019-04-23 16:57:58.710,-6.14,3.38,-5.31,2333,NaN,68,58
2019-04-23 16:57:58.810,-5.26,3.79,-7.16,2291,NaN,-42,-110
2019-04-23 16:57:58.910,-4.71,2.56,-7.78,2256,NaN,-35,7
2019-04-23 16:57:59.010,-5.28,4.05,-7.26,2227,NaN,-29,6
2019-04-23 16:57:59.110,-7.84,4.74,-5.51,2237,NaN,10,39
2019-04-23 16:57:59.210,-4.96,2.08,-8.07,2204,NaN,-33,-43
2019-04-23 16:57:59.310,-3.08,1.46,-9.39,2180,NaN,-24,9


In [532]:
airgo_data['pos_ventilation0'] = np.maximum(np.zeros(airgo_data['deriv1'].shape),airgo_data['deriv1'].values)

In [533]:
airgo_data['pos_ventilation_3s'] = airgo_data['pos_ventilation0'].rolling('3s').sum()*20
airgo_data['pos_ventilation_10s'] = airgo_data['pos_ventilation0'].rolling('10s').sum()*6
airgo_data['pos_ventilation_60s'] = airgo_data['pos_ventilation0'].rolling('60s').sum()
airgo_data['hypo_10_60'] = np.zeros(airgo_data['pos_ventilation_60s'].shape)
airgo_data['hypo_10_60'][airgo_data['pos_ventilation_60s'].values != 0] = np.divide(airgo_data['pos_ventilation_10s'][airgo_data['pos_ventilation_60s'].values != 0] , airgo_data['pos_ventilation_60s'].values[airgo_data['pos_ventilation_60s'].values != 0])

In [534]:
airgo_data['movavg_1s'] = airgo_data['Band'].rolling('1s').mean()
airgo_data['movavg_1_2s'] = airgo_data['Band'].rolling('1200ms').mean()
airgo_data['movavg_2s'] = airgo_data['Band'].rolling('2s').mean()
airgo_data['movavg_3s'] = airgo_data['Band'].rolling('3s').mean()
airgo_data['movavg_5s'] = airgo_data['Band'].rolling('5s').mean()
airgo_data['movavg_10s'] = airgo_data['Band'].rolling('10s').mean()
airgo_data['movavg_60s'] = airgo_data['Band'].rolling('60s').mean()

In [535]:
intersect_idx = np.argwhere(np.diff(np.sign(airgo_data['Band'] - airgo_data['movavg_2s']))).flatten()

In [536]:
intersect_idx[:10]

array([  0,   3,   4,   7,  22,  47,  60,  90, 103, 128], dtype=int64)

In [537]:
smallbreathidx_start = np.diff(intersect_idx)<5
smallbreathidx_end = np.concatenate([[False],smallbreathidx_start[:-1]])
smallbreathidx = smallbreathidx_start | smallbreathidx_end
intersect_idx = intersect_idx[np.concatenate([~smallbreathidx, [True]])]


intersect_idx[:10]

array([ 22,  47,  60,  90, 103, 128, 143, 163, 177, 201], dtype=int64)

In [538]:
smallbreathidx = intersect_idx[np.concatenate([~smallbreathidx_start, [True]])]


IndexError: boolean index did not match indexed array along dimension 0; dimension is 5304 but corresponding boolean dimension is 7952

In [539]:
smallbreathidx_start[:10]

array([ True,  True,  True, False, False, False, False, False, False,
       False])

In [540]:
smallbreathidx_end[:10]

array([False,  True,  True,  True, False, False, False, False, False,
       False])

In [541]:
np.zeros(airgo_data['deriv1'].shape).shape

(100000,)

In [542]:
airgo_data['deriv1'].values.shape

(100000,)

In [15]:
airgo_data['accX_movavg_1s'] = airgo_data['accX'].rolling('1s').mean()
airgo_data['accY_movavg_1s'] = airgo_data['accY'].rolling('1s').mean()
airgo_data['accZ_movavg_1s'] = airgo_data['accZ'].rolling('1s').mean()

In [27]:
plt.close()
f, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, sharex=True)
ax1.plot(airgo_data.index, airgo_data.Band)
ax2.plot(airgo_data.index, airgo_data.accX)
# ax2.plot(airgo_data.index, airgo_data.accX_movavg_1s)
ax3.plot(airgo_data.index, airgo_data.accY)
# ax3.plot(airgo_data.index, airgo_data.accY_movavg_1s)
ax4.plot(airgo_data.index, airgo_data.accZ)
# ax4.plot(airgo_data.index, airgo_data.accZ_movavg_1s)
# plt.legend(['band','movavg_1s','movavg_1_2s','movavg_2s','intersect']) #,'movavg_5s','movavg_10s','movavg_60s'])
# plt.setp(ax1, xticklabels=[])
# plt.setp(ax2, xticklabels=[])
# plt.setp(ax3, xticklabels=[])
# plt.tight_layout()
plt.show()


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [543]:
plt.close()
plt.figure()

plt.plot(airgo_data.index, airgo_data.Band)
plt.plot(airgo_data.index, airgo_data.movavg_1s)
plt.plot(airgo_data.index, airgo_data.movavg_1_2s)
plt.plot(airgo_data.index, airgo_data.movavg_2s)
# plt.plot(airgo_data.index, airgo_data.movavg_3s)
plt.plot(airgo_data.index[intersect_idx], airgo_data.movavg_2s.iloc[intersect_idx], 'ro')
# plt.plot(airgo_data.index[np.where(smallbreathidx_start), airgo_data.movavg_2s.iloc[smallbreathidx_start], 'go')
# plt.plot(airgo_data.index, airgo_data.movavg_5s)
# plt.plot(airgo_data.index, airgo_data.movavg_10s)
# plt.plot(airgo_data.index, airgo_data.movavg_60s)
plt.legend(['band','movavg_1s','movavg_1_2s','movavg_2s','intersect']) #,'movavg_5s','movavg_10s','movavg_60s'])
plt.show()


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [544]:
# Find local peaks
mv1_2 = airgo_data.movavg_1_2s.values
# airgo_data.reset_index(inplace=True)
localmin_idx = airgo_data.movavg_1_2s[(airgo_data.movavg_1_2s.shift(1) > airgo_data.movavg_1_2s) & (airgo_data.movavg_1_2s.shift(-1) > airgo_data.movavg_1_2s)].index.values
localmax_idx = airgo_data.movavg_1_2s[(airgo_data.movavg_1_2s.shift(1) < airgo_data.movavg_1_2s) & (airgo_data.movavg_1_2s.shift(-1) < airgo_data.movavg_1_2s)].index.values



In [545]:
n=2 # number of points to be checked before and after # 0.2 sec
# Find local peaks
airgo_data['min'] = airgo_data.iloc[argrelextrema(airgo_data.Band.values, np.less_equal, order=n)[0]]['Band']
airgo_data['max'] = airgo_data.iloc[argrelextrema(airgo_data.Band.values, np.greater_equal, order=n)[0]]['Band']

# Plot results
plt.figure()
plt.scatter(airgo_data.index, airgo_data['min'], c='r')
plt.scatter(airgo_data.index, airgo_data['max'], c='g')
plt.plot(airgo_data.index, airgo_data['Band'])
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [546]:


# version A
# peaks = find_peaks(airgo_data.Band.values, prominence=25)[0]

# idx_small_diff = np.where(np.diff(peaks) < 12)[0]
# idx_small_diff = [x + np.argmin([airgo_data.Band.values[int(peaks[x])],airgo_data.Band.values[int(peaks[x+1])]]) for x in idx_small_diff]
# peaks = list(set(peaks) - set(peaks[idx_small_diff]))
# peaks.sort()

#version B:
peaks = find_peaks(airgo_data.Band.values, prominence=25, distance = 5, width=3)[0]
print(np.mean(np.diff(peaks)))
print(np.std(np.diff(peaks)))

30.406753878916945
10.987854152174199


In [547]:
plt.close()
plt.figure()

plt.plot(airgo_data.index, airgo_data.Band)
# plt.plot(airgo_data.index, airgo_data.movavg_1s)
# plt.plot(airgo_data.index, airgo_data.movavg_1_2s)
# plt.plot(airgo_data.index, airgo_data.movavg_2s)
# plt.plot(airgo_data.index, airgo_data.movavg_3s)
# plt.scatter(airgo_data.index[localmin_idx-7], airgo_data.Band.iloc[localmin_idx-7], c='r')
plt.scatter(airgo_data.index[peaks], airgo_data.Band.iloc[peaks], c='r', marker ='o')
# plt.scatter(airgo_data.index[peaks2], airgo_data.Band.iloc[peaks2], c='g',marker='<')
# plt.plot(airgo_data.index[np.where(smallbreathidx_start), airgo_data.movavg_2s.iloc[smallbreathidx_start], 'go')
# plt.plot(airgo_data.index, airgo_data.movavg_5s)
# plt.plot(airgo_data.index, airgo_data.movavg_10s)
# plt.plot(airgo_data.index, airgo_data.movavg_60s)
plt.legend(['raw AirGo','peaks detected']) #,'movavg_5s','movavg_10s','movavg_60s'])
plt.show()


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [548]:
airgo_data['peaks'] = 0
airgo_data['peaks'].iloc[peaks] = 1

C:\Users\wg984\.conda\envs\visualization\lib\site-packages\pandas\core\indexing.py:205: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_with_indexer(indexer, value)


In [567]:
airgo_data['rr_10s'] = airgo_data['peaks'].rolling(window='10s').sum()*6
airgo_data['rr_60s'] = airgo_data['peaks'].rolling(window='60s').sum()

In [568]:
normalized = 0
plt.close()
plt.figure()
if normalized:
    plt.plot(airgo_data.index, (airgo_data.Band - np.mean(airgo_data.Band))/np.std(airgo_data.Band)  )
    plt.plot(airgo_data.index, (airgo_data.rr_10s - np.mean(airgo_data.rr_10s))/np.std(airgo_data.rr_10s) )
    plt.plot(airgo_data.index, (airgo_data.rr_60s - np.mean(airgo_data.rr_60s))/np.std(airgo_data.rr_60s)  )
else:
    plt.plot(airgo_data.index, (airgo_data.Band - np.mean(airgo_data.Band))/np.std(airgo_data.Band)*20  )
    plt.plot(airgo_data.index, airgo_data.rr_10s)
    plt.plot(airgo_data.index, airgo_data.rr_60s)
    
plt.legend(['raw AirGo','respiration rate 10sec', 'respiration rate 60sec']) #,'movavg_5s','movavg_10s','movavg_60s'])
plt.show()


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [349]:
airgo_data.index[5]

5

In [352]:
airgo_data.index[1]

1

In [284]:
mini

3        2043.750000
7        2041.250000
47       2038.250000
90       2025.916667
163      2051.333333
            ...     
99826    1632.750000
99863    1641.583333
99907    1628.333333
99951    1633.416667
99991    1646.000000
Name: movavg_1_2s, Length: 3598, dtype: float64

In [285]:
maxi

4        2043.800000
22       2172.333333
61       2140.916667
142      2134.250000
177      2150.583333
            ...     
99801    1687.916667
99840    1697.750000
99877    1704.083333
99922    1703.583333
99967    1707.416667
Name: movavg_1_2s, Length: 3583, dtype: float64

In [105]:
plt.figure()
plt.plot(airgo_data.index, airgo_data.Band - np.mean(airgo_data.Band))
plt.plot(airgo_data.index, airgo_data.deriv1)
# plt.plot(airgo_data.index, airgo_data.deriv2)
plt.plot(airgo_data.index, airgo_data.pos_ventilation_10s)
plt.plot(airgo_data.index, airgo_data.pos_ventilation_60s)
plt.plot(airgo_data.index, airgo_data.hypo_10_60*1000)
plt.legend(['band','deriv1','vent10s','vent60s','hypo_10_60'])
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [78]:
airgo_data.head()

,accX,accY,accZ,Band,CRCStatus,deriv1,deriv2,pos_minute_ventilation,pos_minute_ventilation0,pos_ventilation_10s,pos_ventilation_60s
DateTime,,,,,,,,,,,
2019-04-15 18:11:11.350,-1.99,-1.82,-9.42,2053.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0
2019-04-15 18:11:11.450,-1.94,-1.80,-9.42,2046.0,NaN,-7.0,-7.0,0.0,0.0,0.0,0.0
2019-04-15 18:11:11.550,-2.05,-1.89,-9.45,2039.0,NaN,-7.0,0.0,0.0,0.0,0.0,0.0
2019-04-15 18:11:11.650,-1.91,-1.73,-9.43,2037.0,NaN,-2.0,5.0,0.0,0.0,0.0,0.0
2019-04-15 18:11:11.750,-1.94,-1.84,-9.49,2044.0,NaN,7.0,9.0,7.0,7.0,42.0,7.0


In [79]:
airgo_data.tail()

,accX,accY,accZ,Band,CRCStatus,deriv1,deriv2,pos_minute_ventilation,pos_minute_ventilation0,pos_ventilation_10s,pos_ventilation_60s
DateTime,,,,,,,,,,,
2019-04-16 17:46:34.550,-5.69,-6.85,-4.45,2431.0,NaN,-27.0,-79.0,414.0,0.0,5160.0,6599.0
2019-04-16 17:46:34.650,-5.22,-5.63,-3.49,2463.0,NaN,32.0,59.0,434.0,32.0,5340.0,6631.0
2019-04-16 17:46:34.750,-4.47,-5.68,-4.93,2462.0,NaN,-1.0,-33.0,434.0,0.0,5340.0,6631.0
2019-04-16 17:46:34.850,-6.03,-7.70,-3.91,2415.0,NaN,-47.0,-46.0,434.0,0.0,5340.0,6631.0
2019-04-16 17:46:34.950,-5.38,-6.50,-3.48,2404.0,NaN,-11.0,36.0,434.0,0.0,5340.0,6631.0
